### **Imports**

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_dir = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "src" / "config.py").is_file()
    ),
    None,
)

if project_dir is None:
    raise FileNotFoundError("Open the notebook from the project directory")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.config import MANIFESTS_DIR
from src.processing.embedding_manifest import file_sha256
from src.processing.manifests import CATEGORY_DIRS, read_manifest
from src.processing.sampling import (
    collect_frame_runs,
    evaluate_negative_sampling,
    evaluate_block_sampling,
)

### **Validate the input manifest**

In [2]:
source = json.loads(
    (MANIFESTS_DIR / "sources" / "hf.json").read_text(encoding="utf-8")
)
version = source["annotation_version"]

selection_dir = MANIFESTS_DIR / "embeddings" / version / "within_video"
selection_path = selection_dir / "main.csv"
baseline_path = MANIFESTS_DIR / "baseline" / version / "main.csv"

selection_run = json.loads(
    (selection_dir / "run.json").read_text(encoding="utf-8")
)

if selection_run["source"] != source:
    raise ValueError("Selection source mismatch")

if file_sha256(baseline_path) != selection_run["baseline_sha256"]:
    raise ValueError("Baseline checksum mismatch")

if file_sha256(selection_path) != selection_run["manifest_sha256"]:
    raise ValueError("Selection manifest checksum mismatch")

rows = read_manifest(selection_path)

print(f"Dataset revision: {source['revision']}")
print(f"Manifest checksum: {selection_run['manifest_sha256']}")

Dataset revision: 09c0d83e5c22fc940d87eef5a657ee225d4d5891
Manifest checksum: 76f5ea5eb9af2d18101549aba81c661d8756c55bd7a55607253738527e0aa33f


### **Consecutive negative runs**


Group active negatives by split and source video. Start a new run whenever
original frame numbers are not consecutive. Runs containing one or two
frames are preserved completely.

In [3]:
negative_runs = collect_frame_runs(rows, "negative")

expected_negatives = sum(
    row["split"] in ("val", "test")
    and row["category"] == "negative"
    and not row["excluded_at"]
    for row in rows
)

assert negative_runs["frame_count"].sum() == expected_negatives

run_summary = (
    negative_runs
    .assign(short_run=negative_runs["frame_count"].le(2))
    .groupby("split")
    .agg(
        videos=("video_id", "nunique"),
        runs=("video_id", "size"),
        negative_frames=("frame_count", "sum"),
        protected_short_runs=("short_run", "sum"),
        longest_run=("frame_count", "max"),
    )
    .reindex(["val", "test"], fill_value=0)
)

display(run_summary)
display(
    negative_runs.groupby("split")["frame_count"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90])
    .round(2)
)

,videos,runs,negative_frames,protected_short_runs,longest_run
split,,,,,
val,23,23,304,13,124
test,16,17,341,4,78


,count,mean,std,min,25%,50%,75%,90%,max
split,,,,,,,,,
test,17.0,20.06,24.32,1.0,3.0,7.0,31.0,53.8,78.0
val,23.0,13.22,29.79,1.0,1.0,2.0,7.0,24.4,124.0


### **Run-length distribution**

Count negative runs by length to show how much of each split consists of
isolated frames, short runs, and long consecutive sequences.

In [4]:
length_labels = ["1", "2", "3–5", "6–10", "11–30", "31–100", "101+"]

length_groups = pd.cut(
    negative_runs["frame_count"],
    bins=[0, 1, 2, 5, 10, 30, 100, np.inf],
    labels=length_labels,
)

length_distribution = pd.crosstab(
    negative_runs["split"],
    length_groups,
    dropna=False,
).reindex(
    index=["val", "test"],
    columns=length_labels,
    fill_value=0,
)

display(length_distribution)

frame_count,1,2,3–5,6–10,11–30,31–100,101+
split,,,,,,,
val,10,3,3,2,3,1,1
test,2,2,3,4,1,5,0


### **Compare sampling steps**

In [5]:
sampling_steps = [1, 2, 3, 4]

temporal_summary = evaluate_negative_sampling(
    negative_runs,
    sampling_steps,
)

display(temporal_summary.round(2))

,split,step,videos,runs,input_frames,kept_frames,removed_frames,removed_pct
0,val,1,23,23,304,304,0,0.00
1,val,2,23,23,304,167,137,45.07
2,val,3,23,23,304,121,183,60.20
3,val,4,23,23,304,99,205,67.43
4,test,1,16,17,341,341,0,0.00
5,test,2,16,17,341,183,158,46.33
6,test,3,16,17,341,129,212,62.17
7,test,4,16,17,341,105,236,69.21


## **Temporal sampling of validation and test frames**

### **Block sampling for both categories**

Group frames by source video and category, then split them into consecutive
runs using original frame numbers. Retain one central frame per block.

Compare the same block sizes for positive and negative validation frames.
The selected protocol will be applied unchanged to test.

In [7]:
block_sizes = [5, 10]
category_block_results = []

for category in CATEGORY_DIRS:
    category_runs = collect_frame_runs(rows, category)

    result = evaluate_block_sampling(
        category_runs,
        block_sizes,
        split="val",
    )
    result.insert(0, "category", category)
    category_block_results.append(result)

block_results = pd.concat(
    category_block_results,
    ignore_index=True,
)

block_summary = (
    block_results.groupby(["category", "block_size"])
    .agg(
        videos=("video_id", "size"),
        runs=("runs", "sum"),
        input_frames=("input_frames", "sum"),
        kept_frames=("kept_frames", "sum"),
        removed_frames=("removed_frames", "sum"),
    )
)

block_summary["removed_pct"] = (
    100
    * block_summary["removed_frames"]
    / block_summary["input_frames"]
)

display(block_summary.round(2))

for category in CATEGORY_DIRS:
    print(f"Validation / {category}")

    category_result = block_results.loc[
        block_results["category"] == category
    ]

    retained_by_video = category_result.pivot(
        index="video_id",
        columns="block_size",
        values="kept_frames",
    )
    display(retained_by_video)


videos  runs  input_frames  kept_frames  removed_frames  \
category block_size                                                            
negative 5               23    23           304           74             230   
         10              23    23           304           47             257   
positive 5               84   119          8147         1680            6467   
         10              84   119          8147          875            7272   

                     removed_pct  
category block_size               
negative 5                 75.66  
         10                84.54  
positive 5                 79.38  
         10                89.26

Validation / positive


block_size,5,10
video_id,,
cross_rgb_sky_012,30,15
cross_thermal_sky_035,9,5
cross_thermal_sky_038,43,22
delta_rgb_complex_110,6,3
delta_rgb_field_049,6,4
...,...,...
wing_thermal_sky_563,10,5
wing_thermal_sky_569,17,9
wing_thermal_sky_572,8,4


Validation / negative


block_size,5,10
video_id,,
cross_rgb_sky_012,1,1
delta_rgb_field_049,1,1
delta_rgb_sky_088,1,1
delta_thermal_field_123,1,1
delta_thermal_field_134,1,1
delta_thermal_horizon_187,1,1
delta_thermal_horizon_194,3,2
wing_rgb_field_242,1,1
wing_rgb_field_252,1,1


## **Selected temporal sampling protocol**

Use a block size of **5** for both positive and negative frames.
Within each source video and category, split consecutive frame runs into
non-overlapping blocks and retain one central frame per block. For even
block lengths, retain the earlier central frame.

| Block size | Validation positives | Validation negatives | Total |
|-----------:|---------------------:|---------------------:|------:|
| 5 | 1,680 | 74 | 1,754 |
| 10 | 875 | 47 | 922 |

Block size 5 substantially reduces temporal sampling density while retaining
more observations across each run than block size 10. Every nonempty run
retains at least one representative.

Apply the same fixed rule to test frames and record the selection in a
versioned manifest.